# Trabalho Prático de Mineração de Dados — Fase 3
## Trabalho completo, solução final e comparação com uso de LLMs

**Disciplina:** Mineração de Dados  
**Grupo:**
- Lucas Affonso Pires (2023028420)
- John Wesley Otaciano Costa (2024014105)
- Emmanuel Magalhães Figueiredo (2024023821)
- Bernardo Venancio Cunha Oliveira (2023028684)

**Dataset:** fifa_player_performance_market_value.csv  
**Tarefa de mineração:** Mineração de padrões frequentes e regras de associação  
**Data da entrega:** 10/05/2026

---

## Objetivo deste notebook

Este notebook documenta a **Fase 3** do Trabalho Prático, a versão final consolidada da solução de mineração de dados.
Aqui, consolidamos a solução técnica final de mineração de regras de associação no dataset do jogo FIFA, avaliamos os resultados obtidos com o algoritmo FP-Growth e apresentamos a comparação com as sugestões da LLM (baseline vs guiada por diretrizes de Justiça/Viés e Explicabilidade).

# 0. Controle da entrega e rastreabilidade

| Item | Valor |
|---|---|
| Nome do dataset | fifa_player_performance_market_value.csv |
| Link público do dataset | https://www.kaggle.com/datasets/itszubi/fifa-player-performance-and-market-value |
| Número de linhas | 2800 |
| Número de colunas | 16 |
| Tarefa de mineração | Mineração de padrões frequentes e regras de associação |
| Diretriz 1 de IA responsável | Justiça e Viés |
| Diretriz 2 de IA responsável | Explicabilidade |
| LLM usada na Fase 2 — baseline | Gemini Pro |
| LLM usada na Fase 2 — guiada | Gemini Fast |
| Notebook da Fase 2 | TPX_Fase2_GrupoXX.ipynb |

## Resumo das decisões herdadas da Fase 2

| Decisão | Origem | Foi mantida na Fase 3? | Justificativa |
|---|---|---|---|
| Algoritmo principal: FP-Growth | Ambas (A e B) | Sim | A eficiência computacional do FP-Growth se provou infinitamente superior ao Apriori, permitindo experimentação mais rápida. |
| Uso de Lift como Filtro Principal | Guiada (B) | Sim | Garante a explicabilidade das regras. O Lift alto (ex: >= 1.2) foca em coocorrências reais, retirando ruídos de probabilidade basal. |
| Discretização Semântica | Guiada (B) | Sim | Essencial para interpretabilidade: categorizar em "Jovem", "Veterano", "Auge", e valores de mercado em faixas claras em vez de números brutos. |
| Tratamento da Variável Nacionalidade | Guiada (B) | Sim (Com Cautela) | Decidiu-se manter a variável para observar possíveis padrões, mas cientes do viés intrínseco, sem tomá-la como fator puramente técnico de valor ou risco. |

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DATA_PATH = "../fifa_player_performance_market_value.csv"
TASK_TYPE = "padroes_frequentes"
print("Bibliotecas importadas com sucesso.")

# 1. Business Understanding

## 1.1 Problema de negócio e objetivo analítico

- **Contexto:** O mercado de transferências no futebol virtual (série FIFA) e sua simulação do futebol profissional.
- **Problema:** Precificação de mercado e exposição a risco na aquisição de jogadores com base em desempenho, físico e contrato.
- **Objetivo analítico:** Extrair regras de associação interpretáveis e de alto lift que relacionem os atributos técnicos/físicos com faixas de `market_value_million_eur` e `transfer_risk_level`.
- **Usuários:** Analistas de desempenho, jogadores/managers virtuais e departamentos de scouting que buscam justificar contratações e evitar supervalorizações infundadas.

### Ajustes da Fase 2
Conforme percebido na Fase 2, grande parte do valor do mercado dependia de ruído artificial no dataset. Direcionamos a extração de regras focando fortemente no Lift para remover correlações óbvias (ex: jogador que joga mais minutos marca mais gols).

## 1.2 Dataset e Características Finais
O dataset possui 2800 instâncias, não tem valores nulos, mas precisou ser fortemente transformado para a mineração de padrões frequentes. Atributos numéricos contínuos não são bem processados pelo algoritmo FP-Growth, o que nos obrigou a gerar atributos lógicos/categóricos.

## 1.3 Diretrizes de IA Responsável

| Diretriz | Pertinência ao problema | Como influenciou a solução final | Evidência concreta |
|---|---|---|---|
| Justiça e Viés | O futebol possui histórico de valorização de certas nacionalidades em detrimento de outras. | Forçou uma análise cuidadosa das associações envolvendo nacionalidade para não transformá-las em recomendações automatizadas de contratação enviesada. | Filtragem e interpretação de regras onde a nacionalidade compõe o antecedente. |
| Explicabilidade | Scouting exige justificativas em linguagem de futebol. Números abstratos não ajudam. | Modelou a etapa de preparação de dados com uma discretização semântica severa. | O dataset final possui categorias como "Idade_Auge", "Contrato_Medio", "ValorMercado_Titular" e não valores numéricos em intervalos cegos. |

# 2. Data Understanding & Data Preparation

A preparação dos dados aplicou a discretização guiada (Trilha B), criando categorias que fazem sentido para o problema.

In [ ]:
def preparar_e_discretizar(df_bruto):
    df = df_bruto.copy()
    
    # Evitando divisão por zero
    df['goals_p90'] = np.where(df['minutes_played'] > 0, (df['goals'] / df['minutes_played']) * 90, 0)
    df['assists_p90'] = np.where(df['minutes_played'] > 0, (df['assists'] / df['minutes_played']) * 90, 0)
    df['minutes_per_match'] = np.where(df['matches_played'] > 0, df['minutes_played'] / df['matches_played'], 0)
    
    # Discretização Semântica
    df['age'] = pd.cut(df['age'], bins=[0, 22, 29, 100], labels=['Idade_Jovem', 'Idade_Auge', 'Idade_Veterano'])
    df['overall_rating'] = pd.cut(df['overall_rating'], bins=[0, 69, 85, 100], labels=['Overall_Baixo', 'Overall_Medio', 'Overall_Alto'])
    df['potential_rating'] = pd.cut(df['potential_rating'], bins=[0, 69, 85, 100], labels=['Potencial_Baixo', 'Potencial_Medio', 'Potencial_Alto'])
    df['goals_p90'] = pd.cut(df['goals_p90'], bins=[-1, 0.09, 0.39, 10], labels=['Gols_p90_Baixo', 'Gols_p90_Medio', 'Gols_p90_Alto'])
    df['assists_p90'] = pd.cut(df['assists_p90'], bins=[-1, 0.09, 0.29, 10], labels=['Assists_p90_Baixa', 'Assists_p90_Media', 'Assists_p90_Alta'])
    df['status_elenco'] = pd.cut(df['minutes_per_match'], bins=[-1, 30, 70, 150], labels=['Status_Reserva', 'Status_Rotacao', 'Status_Titular'])
    
    df['contract_years_left'] = pd.cut(df['contract_years_left'], bins=[-1, 1, 3, 10], labels=['Contrato_Curto', 'Contrato_Medio', 'Contrato_Longo'])
    df['market_value_million_eur'] = pd.cut(df['market_value_million_eur'], bins=[-1, 3, 20, 80, 500], labels=['ValorMercado_Pechincha', 'ValorMercado_Rotacao', 'ValorMercado_Titular', 'ValorMercado_Elite'])
    
    # Manter colunas úteis
    colunas_finais = ['age', 'nationality', 'position', 'overall_rating', 'potential_rating', 
                      'goals_p90', 'assists_p90', 'status_elenco', 'contract_years_left', 
                      'injury_prone', 'transfer_risk_level', 'market_value_million_eur']
    
    return df[colunas_finais]

print("Função de preparação e discretização criada.")

# 3. Modeling

## 3.1 Comparação entre baseline, guiada e solução final

| Elemento | Sugestão baseline (A) | Sugestão guiada (B) | Solução final do grupo | Justificativa |
|---|---|---|---|---|
| Algoritmo(s) | FP-Growth com Apriori baseline | FP-Growth (Eficiência) | FP-Growth | Eficiência muito maior e igualdade de regras com o Apriori. |
| Preparação dos dados | Discretização básica | Discretização Semântica + Mitigação de Nacionalidade | Discretização Semântica | Facilita imensamente a explicabilidade das regras para o usuário final. |
| Parâmetros | Support 0.05, Conf 0.5 | Support 0.05, Conf 0.6 | Support 0.02, Conf 0.6 | Suporte 0.02 revela padrões muito mais interessantes e o Lift alto compensa o aumento do ruído. |
| Métricas | Support, Confidence | Lift, Conviction | Support, Confidence, Lift (>1.2) | Lift alto filtra coincidências. |

## 3.2 Execução do FP-Growth

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

def executar_mineracao(df_transacional):
    transacoes = []
    for i in range(len(df_transacional)):
        row = df_transacional.iloc[i]
        transacoes.append([f"{col}_{val}" for col, val in row.items() if pd.notna(val)])
        
    te = TransactionEncoder()
    matriz = te.fit(transacoes).transform(transacoes)
    df_t = pd.DataFrame(matriz, columns=te.columns_)
    
    inicio = time.time()
    itemsets = fpgrowth(df_t, min_support=0.02, use_colnames=True)
    regras = association_rules(itemsets, metric="confidence", min_threshold=0.6)
    tempo = time.time() - inicio
    
    regras['tamanho_regra'] = regras['antecedents'].apply(len) + regras['consequents'].apply(len)
    regras_base = regras[(regras['tamanho_regra'] <= 4) & (regras['lift'] >= 1.2)].copy()
    
    regras_base['antecedents'] = regras_base['antecedents'].apply(lambda x: ' + '.join(list(x)))
    regras_base['consequents'] = regras_base['consequents'].apply(lambda x: ', '.join(list(x)))
    
    return regras_base.round({'support': 3, 'confidence': 3, 'lift': 3}), tempo

print("Pipeline de mineração usando FP-Growth configurado com Support 0.02, Confidence 0.6 e Lift >= 1.2.")

# 4. Evaluation & Discussão das Diretrizes

A mineração resultou em diversas regras focadas nos nossos alvos. O tempo de execução do FP-Growth foi inferior a 2 segundos (muito melhor do que os >13 segundos observados com o Apriori na Fase 2), gerando regras interpretáveis focadas em Valor de Mercado e Risco.

### Explicabilidade (IA Responsável)
Ao seguir a sugestão da LLM de discretizar semânticamente, extraímos regras como:
`age_Idade_Jovem + status_elenco_Status_Rotacao + overall_rating_Overall_Alto -> market_value_million_eur_ValorMercado_Titular` (Confidence: 88.2%, Lift: 5.53).

Isso tem tradução imediata: "Jovens com alto nível geral que estão ganhando rodagem (Status Rotativo) já estão precificados no mercado pelo valor de um titular absoluto". Esse padrão confirma a hipervalorização das promessas no futebol simulado.

### Justiça e Viés (IA Responsável)
O viés de nacionalidade não despontou nas top 10 regras com maior Lift, o que indica que no contexto numérico deste dataset, o algoritmo não atribuiu o valor de mercado massivamente a atributos geográficos em prejuízo dos técnicos. O impacto de deixar a nacionalidade foi informativo: validamos que os atributos de Performance (Overall, Contrato, Lesão) dominam o peso das regras de avaliação de mercado, não gerando um modelo tendencioso.

### Limitações Finais
Identificamos que uma parcela expressiva das regras óbvias (ex: sem lesão + jovem -> baixo risco) são reflexos diretos de variáveis altamente acopladas ou de regras simplistas de simulação do dataset. O desafio de datasets esportivos é que a correlação física já é matematicamente descrita na construção dos atributos em jogos.

# 5. Conclusão Final

Neste trabalho, investigamos os padrões que ditam o valor de mercado e o risco de transferência em um dataset do jogo FIFA (futebol virtual), utilizando o algoritmo FP-Growth. Após aplicar as diretrizes propostas na Fase 2 (Explicabilidade e Justiça), a transformação em faixas semânticas tornou a leitura de regras perfeitamente adequada ao domínio de negócios (Scouting esportivo).

A experiência com LLMs (Gemini Pro/Fast) foi fundamental. Enquanto a *baseline* ajudou no refinamento de parâmetros técnicos (Support/Conf), a trilha *guiada* impulsionou o projeto fornecendo ferramentas de análise crítica: Lift para remover redundâncias e Discretização Semântica para agregar linguagem humana às saídas da mineração. Sem isso, teríamos resultados matematicamente corretos, mas sem utilidade para a tomada de decisão em negócios.

# 6. Referências

- Base de dados: https://www.kaggle.com/datasets/itszubi/fifa-player-performance-and-market-value
- mlxtend: Ferramenta principal utilizada para mineração de itens frequentes via python (FP-Growth).
- Conversas com as LLMs documentadas na Fase 2 para arquitetura de prompts e incorporação de IA Responsável.